# Submission V08 (Nico)

**Goal:** V03 with the month of purchase **split by commercial region** (RG17 / other), the best candidate of `features_region_nico`. Exploratory: it did not pass the 2-SEM rule on the official folds.

| | V03 (Kaggle 0.804) | V08 |
|---|---|---|
| Model | logistic regression | same |
| Preprocessing | median/mode, signed log, robust scaler, one-hot (min 50) | same |
| Features | every column + month of purchase | every column + **month × region** (`3_RG17`, `3_other`, …) instead of the month |
| Feature selection | L1 (`SelectFromModel`, C = 0.05) | same |
| Threshold | best F1 on the out-of-fold predictions of the official folds | same rule |

**Why:** the 4-month cycle is a behaviour of region RG17 (63% of train rows, 53% of test rows): 50% of its purchases fall in peak months and its repeat rate is 0.42 after a peak purchase vs 0.76 before one; the other regions show no cycle. Evidence (`features_region_nico` §3): ROC AUC +0.0062 on the official folds (2/3), **+0.0055 on the 8 half-year folds (7/8, 2.3 SEM)**; honest F1 0.798 vs 0.802.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>Setup</font>](#setup)
* [<font color='#E8800A'>1. The pipeline</font>](#pipeline)
* [<font color='#E8800A'>2. Expected scores from the validation</font>](#expected)
* [<font color='#E8800A'>3. Fit on all train, predict test, submit</font>](#submit)
* [<font color='#E8800A'>4. Publish and upload</font>](#publish)

</div>

# <font color='#E8800A'>Setup</font> <a class="anchor" id="setup"></a>
[Back to TOC](#toc)

`%run` executes `preprocessing_ludovico.ipynb` (about 2 minutes). The feature of `features_region_nico` is restated below exactly as it is defined there.

In [1]:
%%capture
%run ./preprocessing_ludovico.ipynb

In [2]:
from sklearn.feature_selection import SelectFromModel

from src.submissions import check_submission, publish, record_kaggle_score, submission_path, submit

ALWAYS_DROP = ["customer_id", "purchase_date", "random_noise"]
X_full = train.drop(columns=[TARGET])  # identifiers and date are needed to build the feature, then dropped
X_test_full = test.copy()

clientes = pd.read_csv(DATA_RAW / "clientes.csv").set_index("customer_id")
REGION = clientes["commercial_region"].str.extract(r"(RG\d+)")[0].fillna("missing")

# <font color='#E8800A'>1. The pipeline</font> <a class="anchor" id="pipeline"></a>
[Back to TOC](#toc)

`add_features` reads no target and only a static attribute of the customer, so it can run before the split.

In [3]:
def add_features(df):
    """Month of purchase split by commercial region (RG17 / other), then drop the identifiers."""
    group = np.where(df["customer_id"].map(REGION) == "RG17", "RG17", "other")
    df = df.assign(purchase_month=df["purchase_date"].dt.month.astype(str) + "_" + group)
    return df.drop(columns=[c for c in ALWAYS_DROP if c in df.columns])


def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))


l1_selector = SelectFromModel(
    make_pipeline(StandardScaler(), LogisticRegression(l1_ratio=1, solver="saga", C=0.05, max_iter=5000)),
    importance_getter="named_steps.logisticregression.coef_",
)


def v08_pipeline():
    """Logistic regression: month x region -> linear recipe -> L1 selection -> model."""
    steps = linear_recipe(LogisticRegression(max_iter=2000)).set_params(
        prep__num__transform=FunctionTransformer(signed_log1p, feature_names_out="one-to-one")).steps
    return Pipeline([("features", FunctionTransformer(add_features)), *steps[:-1],
                     ("select", clone(l1_selector)), steps[-1]])


print("test rows in RG17:", (X_test_full["customer_id"].map(REGION) == "RG17").mean().round(3))
add_features(X_test_full)["purchase_month"].value_counts().sort_index().to_frame("test rows").T

test rows in RG17: 0.535


purchase_month,10_RG17,10_other,11_RG17,11_other,12_RG17,12_other,1_RG17,1_other,2_RG17,2_other,3_RG17,3_other,4_RG17,4_other,5_RG17,5_other,6_RG17,6_other,7_RG17,7_other,8_RG17,8_other,9_RG17,9_other
test rows,41,21,62,47,10,25,33,30,55,28,21,8,22,31,9,12,72,39,138,48,39,104,26,65


# <font color='#E8800A'>2. Expected scores from the validation</font> <a class="anchor" id="expected"></a>
[Back to TOC](#toc)

Same rules as every submission, on the **official folds**: final threshold = best F1 on the out-of-fold predictions of all folds; honest F1 = threshold chosen on folds 1–2, scored on fold 3 (logged as `CV F1`; so far it predicted Kaggle within ±0.002). The assertion checks that the ROC AUC is that of `features_region_nico` §3 (0.8019).

In [4]:
early = np.concatenate([va for _, va in official_folds[:-1]])
last = official_folds[-1][1]

scores, oof = evaluate(v08_pipeline(), official_folds, X_full, y)
covered = oof.notna()
final_threshold, f1_curve = best_threshold(y[covered], oof[covered])
t_early, _ = best_threshold(y.iloc[early], oof.iloc[early])
expected = pd.Series({"ROC AUC": scores["ROC AUC"].mean(),
                      "final threshold": final_threshold,
                      "threshold folds 1-2": t_early,
                      "honest F1 (fold 3)": f1_score(y.iloc[last], oof.iloc[last] >= t_early)})

assert abs(expected["ROC AUC"] - 0.8019) < 5e-4, expected["ROC AUC"]
expected.round(4)

ROC AUC                0.8019
final threshold        0.4300
threshold folds 1-2    0.4300
honest F1 (fold 3)     0.7985
dtype: float64

In [5]:
# The out-of-fold F1 around the chosen threshold (the curve is usually flat near its maximum)
f1_curve.loc[0.30:0.52].to_frame("OOF F1 (official folds)").T.round(4)

,0.30,0.31,0.32,0.33,0.34,0.35,0.36,0.37,0.38,0.39,0.40,0.41,0.42,0.43,0.44,0.45,0.46,0.47,0.48,0.49,0.50,0.51,0.52
OOF F1 (official folds),0.7551,0.7558,0.7611,0.7606,0.7609,0.7599,0.7597,0.7604,0.7622,0.7599,0.7612,0.7615,0.7624,0.7626,0.7607,0.7584,0.7543,0.753,0.7506,0.7516,0.7478,0.749,0.7493


# <font color='#E8800A'>3. Fit on all train, predict test, submit</font> <a class="anchor" id="submit"></a>
[Back to TOC](#toc)

The pipeline is refitted on **all** clean train rows (the L1 selector picks its columns again), predicts test, and the final threshold turns probabilities into 0/1. `submit` assigns the next free version number and logs it.

In [6]:
final_model = v08_pipeline().fit(X_full, y)
proba = final_model.predict_proba(X_test_full)[:, 1]
prediction = pd.DataFrame({"ID": test.index, TARGET: (proba >= expected["final threshold"]).astype(int)})
check_submission(prediction)

kept = final_model.named_steps["select"].get_support()
names = np.array([n.split("__")[-1] for n in final_model.named_steps["prep"].get_feature_names_out()])
print(f"L1 kept {kept.sum()} of {len(kept)} columns on all train")
print("month x region levels kept:", ", ".join(n.replace("purchase_month_", "") for n in names[kept] if n.startswith("purchase_month")))

v03 = pd.read_csv(submission_path(3))
pd.Series({"share of 1s (V08)": prediction[TARGET].mean(),
           "share of 1s (V03)": v03[TARGET].mean(),
           "same label as V03": (prediction[TARGET].to_numpy() == v03[TARGET].to_numpy()).mean()}).round(3)

L1 kept 27 of 96 columns on all train
month x region levels kept: 10_other, 11_RG17, 11_other, 1_RG17, 3_RG17, 4_RG17, 5_RG17, 7_RG17, 7_other, 8_other, 9_RG17, 9_other


share of 1s (V08)    0.765
share of 1s (V03)    0.736
same label as V03    0.925
dtype: float64

In [7]:
version = submit(
    prediction,
    notebook="submission_v08_nico",
    model="logistic regression + linear recipe + month x region (RG17/other) + L1 selection (C=0.05)",
    cv_f1=expected["honest F1 (fold 3)"],
    notes=(f"EXPLORATORY (did not pass the 2-SEM rule): V03 with the month split by commercial region; "
           f"threshold {expected['final threshold']:.2f} from time CV + gap; CV ROC AUC {expected['ROC AUC']:.3f}; "
           f"8 half-year folds +0.0055 ROC AUC vs V03 (7/8); {prediction[TARGET].mean():.0%} predicted 1"),
)
version

V08 saved as submissions\ML_Group_41_Version08.csv and logged:
  Version: 08 | Date: 2026-10-24 | Author: Bosch | Notebook / model: submission_v08_nico · logistic regression + linear recipe + month x region (RG17/other) + L1 selection (C=0.05) | CV F1: 0.798 | Notes: EXPLORATORY (did not pass the 2-SEM rule): V03 with the month split by commercial region; threshold 0.43 from time CV + gap; CV ROC AUC 0.802; 8 half-year folds +0.0055 ROC AUC vs V03 (7/8); 76% predicted 1
Next: publish(8) to reserve the version, then upload the file on Kaggle.


8

# <font color='#E8800A'>4. Publish and upload</font> <a class="anchor" id="publish"></a>
[Back to TOC](#toc)

**Reserve the version on GitHub** (commit + push of the CSV and the log only), **then** upload the printed file on Kaggle:

In [8]:
# version = publish(version)

After the upload, record the public score and share it:

In [9]:
# record_kaggle_score(version, 0.0)  # <- public score of V08
# publish(version)

### How to read the result
- **V08 vs V03 (0.804):** the only change is the month split by region, unless the threshold moved (see §2 and the share of 1s in §3).
- The honest F1 (0.798) predicts a score close to V03's; the public leaderboard is noisy (±0.02–0.03): read differences under ~0.01 as ties.